In [1]:
# ============================================================
# IMPORTS
# ============================================================

import pandas as pd
import warnings
import sys
import os

warnings.filterwarnings("ignore")
sys.path.append(os.path.abspath(".."))

from functions.tools import configure_notebook_display, load_raw_datasets

configure_notebook_display()

In [2]:
# ============================================================
# LOADING DATASETS
# ============================================================

df_meta, df_readings = load_raw_datasets()

In [3]:
# ============================================================
# DATE PARSING
# ============================================================

df_readings["parsed_date"] = pd.to_datetime(
    df_readings["date"],
    format="mixed",
    dayfirst=True)

df_meta["parsed_sowing_date"] = pd.to_datetime(
    df_meta["sowing_date"])

In [4]:
# ============================================================
# SORTING DATA FOR TEMPORAL ANALYSIS
# ============================================================

df_readings = df_readings.sort_values(
    by=["parcel_id", "parsed_date"])

In [5]:
# ============================================================
# TEMPORAL COVERAGE SUMMARY
# ============================================================

parcel_temporal_summary = (
    df_readings
    .groupby("parcel_id")
    .agg(
        first_date=("parsed_date", "min"),
        last_date=("parsed_date", "max"),
        total_records=("parsed_date", "count"))
    .reset_index())

parcel_temporal_summary["date_span_days"] = (
    parcel_temporal_summary["last_date"] -
    parcel_temporal_summary["first_date"]
    ).dt.days

parcel_temporal_summary["expected_daily_records"] = (
    parcel_temporal_summary["date_span_days"] + 1)

parcel_temporal_summary["missing_days"] = (
    parcel_temporal_summary["expected_daily_records"] -
    parcel_temporal_summary["total_records"])

In [6]:
# ============================================================
# PARCELS WITH MOST MISSING DAYS
# ============================================================

top_missing_days = (
    parcel_temporal_summary
    .sort_values(
        by="missing_days",
        ascending=False)
    .head(5))

display(top_missing_days)

,parcel_id,first_date,last_date,total_records,date_span_days,expected_daily_records,missing_days
25,PARCEL_098,2026-01-09,2026-11-03,20,298,299,279
26,PARCEL_099,2026-01-03,2026-10-04,20,274,275,255
21,PARCEL_022,2026-01-02,2026-12-05,128,337,338,210
16,PARCEL_017,2026-01-02,2026-12-05,130,337,338,208
4,PARCEL_005,2026-01-01,2026-12-05,132,338,339,207


Most parcels contain roughly ~130 observations, while `PARCEL_098` and `PARCEL_099` contain significantly fewer records, indicating incomplete observational coverage.

In [7]:
# ============================================================
# GAP DAYS CALCULATION
# ============================================================

df_readings["previous_date"] = (
    df_readings
    .groupby("parcel_id")["parsed_date"]
    .shift(1))

df_readings["gap_days"] = (
    df_readings["parsed_date"] -
    df_readings["previous_date"]
    ).dt.days

In [8]:
# ============================================================
# LARGEST GAPS PER PARCEL
# ============================================================

largest_gaps = (
    df_readings
    .groupby("parcel_id")["gap_days"]
    .max()
    .reset_index())

largest_gaps.columns = ["parcel_id", "largest_gap_days"]

largest_gaps = largest_gaps.sort_values(
    by="largest_gap_days",
    ascending=False
    ).head(10)

display(largest_gaps)

,parcel_id,largest_gap_days
25,PARCEL_098,90.0
26,PARCEL_099,63.0
2,PARCEL_003,32.0
17,PARCEL_018,32.0
18,PARCEL_019,31.0
21,PARCEL_022,31.0
24,PARCEL_025,31.0
13,PARCEL_014,31.0
15,PARCEL_016,31.0
9,PARCEL_010,31.0


Most parcels follow monthly observation intervals, while `PARCEL_098` and `PARCEL_099` show unusually large temporal discontinuities of 90 and 63 days respectively

In [9]:
# ============================================================
# LARGEST VALID NDVI JUMPS
# ============================================================

valid_ndvi = df_readings[
    (df_readings["ndvi_value"] >= -1) &
    (df_readings["ndvi_value"] <= 1)
    ].copy()

valid_ndvi["previous_ndvi"] = (
    valid_ndvi
    .groupby("parcel_id")["ndvi_value"]
    .shift(1))

valid_ndvi["ndvi_change"] = (
    valid_ndvi["ndvi_value"] -
    valid_ndvi["previous_ndvi"])

valid_ndvi["abs_change"] = (
    valid_ndvi["ndvi_change"].abs())

largest_valid_ndvi_jumps = (
    valid_ndvi
    .sort_values(
        by="abs_change",
        ascending=False
    )
    .head(10))

display(
    largest_valid_ndvi_jumps[
        [
            "parcel_id",
            "parsed_date",
            "ndvi_value",
            "previous_ndvi",
            "ndvi_change"
        ]])

,parcel_id,parsed_date,ndvi_value,previous_ndvi,ndvi_change
2340,PARCEL_023,2026-01-04,0.908,-0.060,0.968
995,PARCEL_019,2026-12-01,0.014,0.890,-0.876
2369,PARCEL_023,2026-01-04,0.106,0.908,-0.802
2191,PARCEL_020,2026-05-02,0.839,0.048,0.791
1989,PARCEL_020,2026-05-02,0.048,0.834,-0.786
1099,PARCEL_022,2026-03-01,0.047,0.812,-0.765
320,PARCEL_001,2026-11-01,0.141,0.878,-0.737
1069,PARCEL_009,2026-06-04,0.840,0.110,0.730
77,PARCEL_019,2026-12-05,0.735,0.014,0.721
703,PARCEL_002,2026-10-01,0.236,0.954,-0.718


Here we can further solidify that parcel 098 and 099 are outliers in our readings. As they do not have any outages - no erroneous status, reporting date is not in range with others. What preprocessing to apply to these two parcels will be looked on in next analysis.

In [10]:
# Updated tracker from notebook 02
issue_tracker = pd.DataFrame({

    "column": [
        "parcel_id",
        "crop_type",
        "sowing_date",
        "date",
        "ndvi_value",
        "sensor_status",
        "parcel_id",
        "parcel_id",
        
    ],

    "issue_identified": [
        "Parcel inconsistency between metadata and readings",
        "Imbalanced crop distribution",
        "Stored as string instead of datetime",
        "Multiple date formats within the same column",
        "NDVI values outside valid biological range [-1, 1]",
        "Dirty categorical labels and missing statuses",
        "Incomplete parcel coverage",
        "Temporal continuity gaps"

    ],

    "prevalence": [
        "3 parcels absent in readings and 2 absent in metadata",
        "Sugarcane dominates by ~ 70% of all parcels",
        "Entire column",
        "3 types of formats detected",
        "104 invalid records detected",
        "Whitespace, casing issues and 137 missing values detected",
        "Parcel 098 and 099 show only 20 record counts, while other have ~130",
        "Large temporal gaps for parcel 098 and 099, 90 and 63 resp"
    ],

    "severity": [
        "High",
        "Low",
        "Medium",
        "High",
        "High",
        "Medium",
        "High",
        "Medium"

    ]
})

display(issue_tracker)

,column,issue_identified,prevalence,severity
0,parcel_id,Parcel inconsistency between metadata and readings,3 parcels absent in readings and 2 absent in metadata,High
1,crop_type,Imbalanced crop distribution,Sugarcane dominates by ~ 70% of all parcels,Low
2,sowing_date,Stored as string instead of datetime,Entire column,Medium
3,date,Multiple date formats within the same column,3 types of formats detected,High
4,ndvi_value,"NDVI values outside valid biological range [-1, 1]",104 invalid records detected,High
5,sensor_status,Dirty categorical labels and missing statuses,"Whitespace, casing issues and 137 missing values detected",Medium
6,parcel_id,Incomplete parcel coverage,"Parcel 098 and 099 show only 20 record counts, while other have ~130",High
7,parcel_id,Temporal continuity gaps,"Large temporal gaps for parcel 098 and 099, 90 and 63 resp",Medium
